In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

In [ ]:
file_path = 'data/insurance.csv'

insurance = pd.read_csv(file_path, encoding='unicode_escape')

In [5]:
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,Southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552


# Teil 1 – Das Business-Problem

Ich arbeite für eine Krankenversicherung.

Die Firma muss entscheiden, wie viel jeder Kunde zahlt (den Beitrag).

Wenn der Preis zu niedrig ist, verliert die Firma Geld, weil die Arztkosten höher sind als die Zahlungen. 💰⬇

Wenn der Preis zu hoch ist, gehen die Kunden zur Konkurrenz. ❌

**Die Frage ist also: „Wie viel wird uns dieser neue Kunde wahrscheinlich kosten?“**

Das Modell beantwortet sie mit Informationen, die wir vor dem Vertrag kennen: Alter, BMI, Raucher oder nicht, Kinder und Region.

# Schritt 1 – Die schmutzigen Daten anschauen

In [8]:
print(insurance.isna().sum()) # was in den Daten schmutzig ist?
for col in ["sex", "smoker", "region"]:
    print(col, insurance[col].unique())

print(insurance[["age", "children"]].describe())

age         66
sex         66
bmi         66
children    66
smoker      66
region      66
charges     54
dtype: int64
sex ['female' 'male' 'woman' 'F' 'man' nan 'M']
smoker ['yes' 'no' nan]
region ['southwest' 'Southeast' 'southeast' 'northwest' 'Northwest' 'Northeast'
 'northeast' 'Southwest' nan]
               age     children
count  1272.000000  1272.000000
mean     35.214623     0.948899
std      22.478251     1.303532
min     -64.000000    -4.000000
25%      24.750000     0.000000
50%      38.000000     1.000000
75%      51.000000     2.000000
max      64.000000     5.000000


# Schritt 2 – Die Daten bereinigen

In [9]:
df = insurance.copy()

# Dasselbe, unterschiedlich geschrieben
# sex hat 6 Schreibweisen für 2 Werte: female, woman, F und male, man, M.
df["sex"] = df["sex"].str.lower().replace({"m": "male", "man": "male",
                                           "f": "female", "woman": "female"})
df["smoker"] = df["smoker"].str.lower()

# region benutzt Klein- und Großschreibung, zum Beispiel Southeast und southeast.
# Ohne Bereinigung würde das Modell denken, dass Southeast und southeast zwei verschiedene Regionen sind.
df["region"] = df["region"].str.lower()

# Entfernt "$" und charges ist Text, keine Zahl
df["charges"] = df["charges"].astype(str).str.replace("$", "", regex=False).astype(float)

# Unmögliche Werte: Das sind Eingabefehler, vielleicht ein aus Versehen getippter Minuszeichen.
# age hat ein Minimum von -64. Niemand ist minus 64 Jahre alt.
df["age"] = df["age"].abs()
df = df[df["age"] > 0]
# children hat ein Minimum von -4. Niemand hat minus 4 Kinder.
df["children"] = df["children"].abs()

# Entfernt leerzeilen
df = df.dropna()
print(df.shape)
print(df["sex"].unique(), df["region"].unique())


# Es gibt keine einzige richtige Antwort. Wichtig ist, dass du bewusst entscheidest und erklären kannst, warum.

(1207, 7)
['female' 'male'] ['southwest' 'southeast' 'northwest' 'northeast']


# Schritt 3 – Text in Zahlen umwandeln und das Modell trainieren

In [10]:
from sklearn.linear_model import LinearRegression

X = pd.get_dummies(df.drop(columns="charges"), drop_first=True)
y = df["charges"]

model = LinearRegression()
r2_score = cross_val_score(model, X, y, cv=5, scoring="r2").mean()
print("R²:", round(r2_score, 3))

model.fit(X, y)
print(pd.Series(model.coef_, index=X.columns).round(0).sort_values())

R²: 0.744
region_southwest     -844.0
region_southeast     -823.0
region_northwest     -271.0
sex_male             -263.0
age                   259.0
bmi                   326.0
children              447.0
smoker_yes          23941.0
dtype: float64


Das Modell erklärt etwa 75 % davon, warum manche Kunden teuer und andere günstig sind.

Jede Zahl sagt dir, wie stark sich die jährlichen Kosten ändern, wenn sich nur dieser eine Faktor ändert und alles andere gleich bleibt.

smoker_yes = +23,941 (Raucher kostet etwa $24,000 pro Jahre.)

children = +447 (jedes Kind fügt etwa $450 hinzu.)

nortwest ist am teuersten

kein Unterschied zwischen Mann und Frau


# Schritt 4 – Für neue Kunden vorhersagen

In [ ]:
file_path_validation = 'data/validation_dataset.csv'

validation_data = pd.read_csv(file_path_validation)

val = validation_data.copy()
val["sex"] = val["sex"].str.lower().replace({"m": "male", "man": "male",
                                             "f": "female", "woman": "female"})
val["smoker"] = val["smoker"].str.lower()
val["region"] = val["region"].str.lower()

X_val = pd.get_dummies(val, drop_first=True).reindex(columns=X.columns, fill_value=0)

validation_data["predicted_charges"] = model.predict(X_val)
# clip(lower=1000) ist die Business-Regel aus der Aufgabe: Jeder Kunde zahlt mindestens 1.000.
validation_data["predicted_charges"] = validation_data["predicted_charges"].clip(lower=1000)
validation_data.head()

,age,sex,bmi,children,smoker,region,predicted_charges
0,18.0,female,24.090000,1.0,no,southeast,1000.000000
1,39.0,male,26.410000,0.0,yes,northeast,30800.081857
2,27.0,male,29.150000,0.0,yes,southeast,27761.777100
3,71.0,male,65.502135,13.0,yes,southeast,56804.280420
4,28.0,male,38.060000,0.0,no,southeast,6981.903497


Notzen:

0. In den letzten Projekten haben wir den RMSE benutzt, also den durchschnittlichen Fehler in kW oder Fahrrädern.
1. R² ist anders. Er zeigt, wie viel von den Unterschieden zwischen den Kunden das Modell erklären kann.
2. R² = 0 bedeutet, dass das Modell nichts weiß.
3. R² = 1 bedeutet, dass das Modell perfekt ist.
4. R² = 0,75 bedeutet, dass das Modell 75 % davon erklärt, warum manche Kunden teuer und andere günstig sind.
